In [1]:
import polars as pl
from pathlib import Path
import geopandas as gpd
import h3
from google.colab import drive
from shapely.geometry import Point
import ee
import numpy as np
from shapely import STRtree
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor


In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
base = Path('/content/drive/MyDrive')

In [ ]:
df = pl.read_csv(base / 'renewably_wind' / 'final_df_processed.csv')
df.describe()

In [ ]:
def explode_to_children(df: pl.DataFrame, target_res: int) -> pl.DataFrame:
    """Split each H3 cell into children at target_res."""
    rows = []
    for h3_index in df["h3_index"].to_list():
        for child in h3.cell_to_children(h3_index, res=target_res):
            lat, lng = h3.cell_to_latlng(child)
            rows.append({
                "h3_index": child,
                "parent_h3": h3_index,
                "lat": lat,
                "lng": lng,
            })
    return pl.DataFrame(rows)

df = explode_to_children(df, 9)
df.head()

## Wind turbine geojson

In [ ]:
turbine_gdf = gpd.read_file(base / 'renewably_wind' / 'wind_us_farms.geojson')

In [ ]:
turbine_gdf = turbine_gdf.to_crs(epsg=4326)
centroids = turbine_gdf.geometry.to_crs(3857).centroid.to_crs(4326)
turbine_gdf['longitude'] = centroids.x
turbine_gdf['latitude'] = centroids.y


In [ ]:
turbine_gdf = turbine_gdf[['fid', 'longitude', 'latitude']]
turbine_gdf.shape

In [ ]:
res = 9

turbine_cells = (
    pl.from_records(
        [
            (h3.latlng_to_cell(lat, lon, res),)
            for lat, lon in zip(
                turbine_gdf["latitude"],
                turbine_gdf["longitude"],
            )
        ],
        schema=["h3_index"],
        orient="row"
    )
    .unique()
)


In [ ]:
df = df.join(
    turbine_cells.with_columns(pl.lit(True).alias("has_turbine")),
    on="h3_index",
    how="left",
).with_columns(pl.col("has_turbine").fill_null(False))

In [ ]:
df.head()

In [ ]:
df['has_turbine'].value_counts()


In [ ]:
n = df['has_turbine'].value_counts().filter(~pl.col("has_turbine"))["count"].item() * .07

true_df = df.filter(pl.col("has_turbine"))
false_df = df.filter(~pl.col("has_turbine")).sample(n=n, seed=42)

df_balanced = pl.concat([true_df, false_df])

In [ ]:
df_balanced.head()
df_balanced['has_turbine'].value_counts()


In [ ]:
df_balanced.head()

### Download Data Sets

In [4]:
ee.Authenticate()
ee.Initialize(project='renewably')

In [ ]:
gdf = gpd.GeoDataFrame(
    df_balanced, 
    geometry=gpd.points_from_xy(df_balanced["lng"], df_balanced["lat"]),
    crs="EPSG:4326"
)

gdf.to_csv(base / 'renewably_wind' / 'points.csv', index=False)

In [ ]:
# Code to generate terrain features on earth engine

points = ee.FeatureCollection('projects/renewably/assets/points')

# Sources
nlcd = ee.ImageCollection('USGS/NLCD_RELEASES/2021_REL/NLCD').first()
nlcd_landcover = nlcd.select(['landcover'])
nlcd_impervious = nlcd.select(['impervious'])
elevation = ee.Image('USGS/SRTMGL1_003').select(['elevation'])
slope_img = ee.Terrain.slope(elevation).rename('slope')

# ERA5 monthly mean — 10m wind components
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
    .filterDate('2020-01-01', '2023-12-31') \
    .select(['u_component_of_wind_10m', 'v_component_of_wind_10m'])

# Compute wind speed magnitude, then average across all months
def wind_speed(img):
    return img.expression(
        'sqrt(u**2 + v**2)',
        {'u': img.select('u_component_of_wind_10m'),
         'v': img.select('v_component_of_wind_10m')}
    ).rename('wind_speed')

mean_wind = era5.map(wind_speed).mean()

# OpenLandMap soil taxonomy great groups
soil = ee.Image('OpenLandMap/SOL/SOL_GRTGROUP_USDA-SOILTAX_C/v01') \
    .select(['grtgroup']).rename('soil_type')
    

# --- Protected areas / restrictions ---

# USGS PAD-US (Protected Areas Database) — includes national parks,
# wilderness, wildlife refuges, military, tribal lands
# GAP Status 1 & 2 = strongest protection (no development)
padus = ee.FeatureCollection('USGS/GAP/PAD-US/v20/designation')

# Rasterize: 1 if inside any protected area, 0 otherwise
protected = padus.reduceToImage(
    properties=[],
    reducer=ee.Reducer.countEvery()
).gt(0).rename('protected_area').unmask(0)

# WDPA (World Database on Protected Areas) — another option with
# IUCN categories, good for cross-referencing
wdpa = ee.FeatureCollection('WCMC/WDPA/current/polygons')
in_wdpa = wdpa.reduceToImage(
    properties=[],
    reducer=ee.Reducer.countEvery()
).gt(0).rename('in_wdpa').unmask(0)

# Population density
ghsl = ee.Image('JRC/GHSL/P2023A/GHS_POP/2020') \
    .select('population_count') \
    .rename('pop_density')


stacked = (nlcd_landcover
    .addBands(nlcd_impervious)
    .addBands(elevation)
    .addBands(slope_img)
    .addBands(mean_wind)
    .addBands(soil)
    .addBands(protected)
    .addBands(in_wdpa)
    .addBands(ghsl)
)

results = stacked.reduceRegions(
    collection=points,
    reducer=ee.Reducer.first(),
    scale=30
)

task = ee.batch.Export.table.toAsset(
    collection=results,
    description='terrain_features_with_protected',
    assetId='projects/renewably/assets/terrain_features_with_protected'
)
task.start()

In [5]:
earthengine_df = pl.read_csv(base / 'renewably_wind' / 'terrain_features_with_protected.csv')
earthengine_df.head()


system:index,0,1,2,3,4,elevation,impervious,in_wdpa,landcover,pop_density,protected_area,slope,soil_type,wind_speed,.geo
str,str,str,f64,f64,bool,i64,str,i64,str,f64,i64,f64,str,f64,str
"""0000000000000001c29f""","""890e4904ac7ffff""","""880e4904adfffff""",49.171908,-67.825001,false,0,null,0,null,-200.0,0,0.0,null,null,"""{""type"":""Point"",""coordinates"":…"
"""0000000000000001ac82""","""890e49058c3ffff""","""880e49058dfffff""",49.196495,-67.878989,false,0,null,0,null,-200.0,0,0.0,null,null,"""{""type"":""Point"",""coordinates"":…"
"""0000000000000001d3af""","""890e49058c7ffff""","""880e49058dfffff""",49.193563,-67.876717,false,0,null,0,null,-200.0,0,0.0,null,null,"""{""type"":""Point"",""coordinates"":…"
"""00000000000000015d81""","""890e490ce3bffff""","""880e490ce3fffff""",49.187634,-68.057107,false,0,null,0,null,-200.0,0,0.0,null,null,"""{""type"":""Point"",""coordinates"":…"
"""00000000000000010782""","""890e491648bffff""","""880e491649fffff""",49.423621,-67.558907,false,134,null,0,null,0.0,0,0.0,null,1.739339,"""{""type"":""Point"",""coordinates"":…"


In [6]:
# Rename for your functions
earthengine_df = earthengine_df.rename({
    '0': 'h3_index',
    '1': 'parent_h3',
    '2': 'lat',
    '3': 'lng',
    '4': 'has_turbine',
    'landcover': 'land_type',
    'elevation': 'elevation_m',
    'slope': 'slope_deg'
})

earthengine_df = earthengine_df.drop(['system:index', '.geo', 'parent_h3'])

In [7]:
earthengine_df.describe()

statistic,h3_index,lat,lng,has_turbine,elevation_m,impervious,in_wdpa,land_type,pop_density,protected_area,slope_deg,soil_type,wind_speed
str,str,f64,f64,f64,f64,str,f64,str,f64,f64,f64,str,f64
"""count""","""147646""",147646.0,147646.0,147646.0,129613.0,"""117570""",147646.0,"""117570""",147646.0,147646.0,129608.0,"""122899""",124697.0
"""null_count""","""0""",0.0,0.0,0.0,18033.0,"""30076""",0.0,"""30076""",0.0,0.0,18038.0,"""24747""",22949.0
"""mean""",null,37.497017,-96.517351,0.435359,650.843172,null,0.03104,null,-29.933342,0.061715,4.087549,null,1.456466
"""std""",null,6.29246,13.766874,null,562.222655,null,0.173428,null,71.57922,0.240638,5.070002,null,0.597872
"""min""","""890e490106fffff""",24.499213,-124.99804,0.0,-71.0,"""0""",0.0,"""11""",-200.0,0.0,0.0,"""1""",0.232127
"""25%""",null,32.682051,-103.279583,null,258.0,null,0.0,null,0.0,0.0,1.453802,null,1.046623
"""50%""",null,37.714823,-97.935445,null,466.0,null,0.0,null,0.0,0.0,2.618975,null,1.325431
"""75%""",null,42.633748,-88.058303,null,884.0,null,0.0,null,0.0,0.0,4.580196,null,1.830702
"""max""","""89509ed68bbffff""",49.501849,-65.999167,1.0,4132.0,"""99""",1.0,"""95""",340.856201,1.0,63.805427,"""99""",6.097871


In [8]:
# Fix nulls
earthengine_df = earthengine_df.with_columns([
    pl.col("elevation_m").cast(pl.Float64).fill_null(0),
    pl.col("slope_deg").cast(pl.Float64).fill_null(0),
    pl.col("land_type").fill_null("0"),  # "0" = no data / unclassified
    pl.col("impervious").fill_null("0"),
    pl.col("soil_type").fill_null("0"),
])

In [9]:
# Encodings 

# NLCD Land Cover — group into meaningful categories for turbine siting
landcover_encoding = {
    "0":  0,   # No data / unclassified
    "11": 1,   # Open water
    "12": 1,   # Perennial ice/snow
    "21": 2,   # Developed, open space
    "22": 3,   # Developed, low intensity
    "23": 4,   # Developed, medium intensity
    "24": 5,   # Developed, high intensity
    "31": 6,   # Barren land
    "41": 7,   # Deciduous forest
    "42": 8,   # Evergreen forest
    "43": 9,   # Mixed forest
    "52": 10,  # Shrub/scrub
    "71": 11,  # Grassland/herbaceous
    "81": 12,  # Pasture/hay
    "82": 13,  # Cultivated crops
    "90": 14,  # Woody wetlands
    "95": 15,  # Emergent herbaceous wetlands
}

# Impervious  encoding
impervious_encoding = {
    "0":  0,   # Unclassified
    "20": 1,   # Primary road
    "21": 2,   # Secondary road
    "22": 3,   # Tertiary road
    "23": 4,   # Thinned road
    "24": 5,   # Non-road non-energy impervious
    "25": 6,   # Microsoft buildings
    "26": 7,   # LCMAP impervious
    "27": 0,   # Wind turbines → mask to unclassified (LEAKAGE)
    "28": 8,   # Well pads
    "29": 9,   # Other energy production
}

earthengine_df = earthengine_df.with_columns(
    pl.col("impervious")
        .fill_null("0")
        .cast(pl.UInt8)
        .replace(impervious_encoding, default=0),
    pl.col("land_type")
        .fill_null("0")
        .cast(pl.UInt8)
        .replace(landcover_encoding, default=0)
)


/tmp/ipykernel_23500/4085774989.py:43: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  .replace(impervious_encoding, default=0),
/tmp/ipykernel_23500/4085774989.py:47: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  .replace(landcover_encoding, default=0)


In [10]:
earthengine_df.describe()

statistic,h3_index,lat,lng,has_turbine,elevation_m,impervious,in_wdpa,land_type,pop_density,protected_area,slope_deg,soil_type,wind_speed
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64
"""count""","""147646""",147646.0,147646.0,147646.0,147646.0,147646.0,147646.0,147646.0,147646.0,147646.0,147646.0,"""147646""",124697.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""",22949.0
"""mean""",null,37.497017,-96.517351,0.435359,571.351313,0.014799,0.03104,8.649872,-29.933342,0.061715,3.588171,null,1.456466
"""std""",null,6.29246,13.766874,null,568.247667,0.298604,0.173428,5.043619,71.57922,0.240638,4.935218,null,0.597872
"""min""","""890e490106fffff""",24.499213,-124.99804,0.0,-71.0,0.0,0.0,0.0,-200.0,0.0,0.0,"""0""",0.232127
"""25%""",null,32.682051,-103.279583,null,175.0,0.0,0.0,6.0,0.0,0.0,0.92741,null,1.046623
"""50%""",null,37.714823,-97.935445,null,407.0,0.0,0.0,11.0,0.0,0.0,2.249939,null,1.325431
"""75%""",null,42.633748,-88.058303,null,808.0,0.0,0.0,13.0,0.0,0.0,4.092187,null,1.830702
"""max""","""89509ed68bbffff""",49.501849,-65.999167,1.0,4132.0,9.0,1.0,15.0,340.856201,1.0,63.805427,"""99""",6.097871


In [11]:
# Build code-to-order mapping from the class table
soil_code_to_order = {
    0: 0,  # NODATA
    # Alfisols (-alfs)
    1: 1, 2: 1, 4: 1, 6: 1, 7: 1, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1,
    14: 1, 15: 1, 16: 1, 17: 1, 18: 1, 19: 1, 25: 1, 26: 1, 27: 1,
    28: 1, 29: 1, 30: 1, 31: 1, 32: 1, 38: 1, 39: 1, 41: 1, 42: 1,
    43: 1, 44: 1, 45: 1, 46: 1,
    # Andisols (-ands)
    50: 2, 58: 2, 59: 2, 61: 2, 63: 2, 64: 2, 74: 2, 75: 2, 76: 2,
    77: 2, 80: 2,
    # Aridisols (-ids)
    82: 3, 83: 3, 85: 3, 86: 3, 87: 3, 89: 3, 90: 3, 92: 3, 93: 3,
    94: 3, 96: 3, 97: 3, 98: 3, 99: 3, 100: 3, 101: 3, 102: 3, 103: 3,
    104: 3, 105: 3, 107: 3, 110: 3, 111: 3, 112: 3, 113: 3, 114: 3,
    115: 3, 116: 3,
    # Entisols (-ents)
    118: 4, 119: 4, 120: 4, 121: 4, 122: 4, 123: 4, 124: 4, 126: 4,
    131: 4, 133: 4, 134: 4, 135: 4, 136: 4, 137: 4, 138: 4, 139: 4,
    140: 4, 141: 4, 142: 4, 143: 4, 144: 4, 145: 4, 146: 4, 147: 4,
    148: 4, 149: 4, 153: 4, 154: 4, 155: 4,
    # Histosols (-ists)
    179: 5, 180: 5, 181: 5, 182: 5, 183: 5, 184: 5, 185: 5, 186: 5,
    189: 5, 190: 5, 191: 5, 196: 5, 201: 5, 202: 5, 203: 5,
    # Inceptisols (-epts)
    206: 6, 207: 6, 208: 6, 209: 6, 210: 6, 212: 6, 213: 6, 215: 6,
    216: 6, 217: 6, 218: 6, 219: 6, 220: 6, 221: 6, 222: 6, 225: 6,
    226: 6, 228: 6, 229: 6, 230: 6, 231: 6, 233: 6, 234: 6, 235: 6,
    245: 6, 246: 6, 247: 6, 248: 6, 249: 6, 250: 6, 251: 6, 252: 6,
    253: 6, 254: 6, 255: 6, 256: 6,
    # Mollisols (-olls)
    268: 7, 269: 7, 270: 7, 271: 7, 272: 7, 273: 7, 274: 7, 275: 7,
    276: 7, 277: 7, 278: 7, 279: 7, 280: 7, 283: 7, 284: 7, 285: 7,
    286: 7, 287: 7, 289: 7, 290: 7, 291: 7, 292: 7, 294: 7, 296: 7,
    297: 7, 298: 7, 299: 7, 300: 7, 301: 7, 302: 7, 303: 7, 306: 7,
    307: 7, 308: 7, 309: 7, 310: 7, 311: 7, 312: 7,
    # Spodosols (-ods)
    342: 8, 343: 8, 345: 8, 348: 8, 349: 8, 350: 8, 351: 8, 353: 8,
    354: 8, 356: 8, 357: 8, 358: 8, 367: 8, 368: 8,
    # Ultisols (-ults)
    370: 9, 371: 9, 372: 9, 373: 9, 374: 9, 375: 9, 376: 9, 377: 9,
    378: 9, 381: 9, 385: 9, 387: 9, 388: 9, 389: 9, 390: 9, 391: 9,
    396: 9, 399: 9, 401: 9,
    # Vertisols (-erts)
    403: 10, 405: 10, 406: 10, 409: 10, 410: 10, 412: 10, 413: 10,
    414: 10, 415: 10, 417: 10, 418: 10, 419: 10, 420: 10, 422: 10,
    424: 10, 429: 10, 430: 10, 431: 10, 432: 10, 433: 10,
}

earthengine_df = earthengine_df.with_columns(
    pl.col("soil_type")
      .cast(pl.Int32)
      .replace_strict(soil_code_to_order, default=0)
      .cast(pl.UInt8)
)

In [12]:
earthengine_df.describe()

statistic,h3_index,lat,lng,has_turbine,elevation_m,impervious,in_wdpa,land_type,pop_density,protected_area,slope_deg,soil_type,wind_speed
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""147646""",147646.0,147646.0,147646.0,147646.0,147646.0,147646.0,147646.0,147646.0,147646.0,147646.0,147646.0,124697.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22949.0
"""mean""",null,37.497017,-96.517351,0.435359,571.351313,0.014799,0.03104,8.649872,-29.933342,0.061715,3.588171,4.821113,1.456466
"""std""",null,6.29246,13.766874,null,568.247667,0.298604,0.173428,5.043619,71.57922,0.240638,4.935218,3.092176,0.597872
"""min""","""890e490106fffff""",24.499213,-124.99804,0.0,-71.0,0.0,0.0,0.0,-200.0,0.0,0.0,0.0,0.232127
"""25%""",null,32.682051,-103.279583,null,175.0,0.0,0.0,6.0,0.0,0.0,0.92741,1.0,1.046623
"""50%""",null,37.714823,-97.935445,null,407.0,0.0,0.0,11.0,0.0,0.0,2.249939,7.0,1.325431
"""75%""",null,42.633748,-88.058303,null,808.0,0.0,0.0,13.0,0.0,0.0,4.092187,7.0,1.830702
"""max""","""89509ed68bbffff""",49.501849,-65.999167,1.0,4132.0,9.0,1.0,15.0,340.856201,1.0,63.805427,10.0,6.097871


In [13]:
# Change var name
df = earthengine_df

In [14]:
df.shape

(147646, 13)

In [15]:
def proximity_to_transmission_line(df: pl.DataFrame) -> pl.DataFrame:
    """Calculate the distance to the nearest transmission line for each point."""
    transmission_lines = gpd.read_file(base / 'renewably_wind' / 'us_transmission_lines.geojson')
    transmission_lines = transmission_lines.to_crs(epsg=5070)

    points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(df['lng'].to_list(), df['lat'].to_list()),
        crs="EPSG:4326"
    ).to_crs(epsg=5070)

    tree = STRtree(transmission_lines.geometry.values)
    point_geoms = points.geometry.values

    indices, distances_m = tree.query_nearest(point_geoms, return_distance=True)
    input_idx = indices[0]

    min_distances = np.full(len(points), np.inf)
    np.minimum.at(min_distances, input_idx, distances_m)
    distances_km = min_distances / 1000.0

    return df.with_columns(
        pl.Series("transmission_line_dist_km", distances_km)
    )



def proximity_to_road(df: pl.DataFrame) -> pl.DataFrame:
    """Calculate distance to nearest road for each point"""
    road_lines = gpd.read_file(base / 'renewably_wind' / 'tl_2023_us_primaryroads.shp').set_crs(epsg=4269)
    road_lines = road_lines.to_crs(epsg=5070)

    points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(df['lng'].to_list(), df['lat'].to_list()),
        crs="EPSG:4326"
    ).to_crs(epsg=5070)

    tree = STRtree(road_lines.geometry.values)
    point_geoms = points.geometry.values

    indices, distances_m = tree.query_nearest(point_geoms, return_distance=True)
    input_idx = indices[0]

    min_distances = np.full(len(points), np.inf)
    np.minimum.at(min_distances, input_idx, distances_m)
    distances_km = min_distances / 1000.0

    return df.with_columns(
        pl.Series("road_dist_km", distances_km)
    )


def proximity_to_airport(df: pl.DataFrame) -> pl.DataFrame:
    airports = gpd.read_file(base / 'renewably_wind' / 'airports.geojson')
    airports = airports.to_crs(epsg=5070)

    points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(df['lng'].to_list(), df['lat'].to_list()),
        crs="EPSG:4326"
    ).to_crs(epsg=5070)

    tree = STRtree(airports.geometry.values)
    
    _, distances_m = tree.query_nearest(points.geometry.values, return_distance=True)

    distances_km = distances_m / 1000.0

    return df.with_columns(
        pl.Series("airport_dist_km", distances_km)
    )

def wind_speed(df: pl.DataFrame) -> pl.DataFrame:
    wind_speeds = pd.read_csv(base / 'renewably_wind' / 'us_wind_speed_dataset_2022.csv')

    knn = KNeighborsRegressor(n_neighbors=5, weights="distance")
    knn.fit(wind_speeds[["lat", "lon"]], wind_speeds["annual_mean_wind_speed"])

    pdf = df.to_pandas()
    for col in ["wind_speed", "annual_mean_wind_speed"]:
        if col in pdf.columns:
            pdf = pdf.drop(columns=[col])

    pdf["wind_speed"] = knn.predict(pdf[["lat", "lng"]].rename(columns={"lng": "lon"}))

    return pl.from_pandas(pdf)
    

In [16]:
df = proximity_to_road(df)

In [17]:
df = proximity_to_transmission_line(df)

In [18]:
df = wind_speed(df)

In [19]:
df = proximity_to_airport(df)

## Data clean up

In [24]:
df.describe()

statistic,h3_index,lat,lng,has_turbine,elevation_m,impervious,in_wdpa,land_type,pop_density,protected_area,slope_deg,soil_type,road_dist_km,transmission_line_dist_km,wind_speed,airport_dist_km
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""147594""",147594.0,147594.0,147594.0,147594.0,147594.0,147594.0,147594.0,147594.0,147594.0,147594.0,147594.0,147594.0,147594.0,147594.0,147594.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,37.496054,-96.519837,0.43516,571.429414,0.014804,0.031051,8.651158,0.132702,0.06173,3.588433,4.821199,115.889809,81.003022,3.457598,70.667926
"""std""",null,6.292927,13.766615,null,568.2966,0.298656,0.173457,5.044042,2.493569,0.240666,4.935685,3.092322,198.639575,196.349224,0.955293,148.8211
"""min""","""890e490106fffff""",24.499213,-124.99804,0.0,-71.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000012,7.5618e-7,1.153435,0.089089
"""25%""",null,32.679965,-103.281105,null,175.0,0.0,0.0,6.0,0.0,0.0,0.92741,1.0,16.288732,1.374384,2.58492,9.435339
"""50%""",null,37.713488,-97.936938,null,407.0,0.0,0.0,11.0,0.0,0.0,2.250011,7.0,37.221294,3.924614,3.625409,15.398782
"""75%""",null,42.634221,-88.063351,null,808.0,0.0,0.0,13.0,0.0,0.0,4.092713,7.0,106.73545,16.776532,4.327785,28.428874
"""max""","""89509ed68bbffff""",49.501849,-65.999167,1.0,4132.0,9.0,1.0,15.0,340.856201,1.0,63.805427,10.0,1187.079656,1147.270167,6.931906,1083.249241


In [22]:
# make sure data is correct, land types
df = df.filter(~(pl.col("land_type").is_in([5]) & pl.col("has_turbine")))

In [23]:
df = df.with_columns(pl.col("pop_density").clip(lower_bound=0))

In [28]:
df.write_csv(base / 'renewably_wind' / 'final_df_processed_with_proximity.csv')